# Example 12.6.7 / 1: Run CCG on four scenarios

Companion notebook for **12.6.7 Two-Stage Robust Optimization and CCG** in the AMPL lab Documentation, section "Run CCG on four scenarios". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Run CCG on four scenarios

A small instance makes every step visible. Two products need capacity bought at 2 per unit
before demand is known; uncovered demand is met by emergency supply at 3 per unit. The
uncertainty set holds four demand vectors, A `(60, 60)`, B `(100, 40)`, C `(40, 100)` and
D `(80, 80)`, so the oracle can enumerate it exactly. The recourse value is
$Q(x,u)=\sum_p 3\max(u_p-x_p,0)$, and emergency supply gives relatively complete recourse.

The model is the restricted master over `ACTIVE`; `Build` is the first-stage capacity, named as in
Decomposition (AMPL lab documentation, section 12.5). The loop resets `ACTIVE`, solves, reads the lower bound
with `value()`, evaluates the oracle expression with `ampl.get_data(...).to_dict()`, and stops
when the bounds meet. The first maximal scenario breaks oracle ties. The last solve uses all
scenarios, the full deterministic equivalent.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ALL ordered;                        # finite uncertainty set U
set ACTIVE within ALL;                  # scenarios in the restricted master
set PROD;                               # products
param demand {ALL, PROD} >= 0;          # units
param capacity_cost {PROD} >= 0;        # dollars per unit of capacity
param emergency_cost {PROD} >= 0;       # dollars per emergency unit

var Build {PROD} >= 0;                  # first stage, shared
var Emergency {ACTIVE, PROD} >= 0;      # recourse copy per active scenario
var Eta >= 0;                           # worst recourse cost so far

minimize MasterCost:                    # capacity cost plus worst active recourse
    sum {p in PROD} capacity_cost[p] * Build[p] + Eta;

# Recourse covers each active scenario's demand.
subject to Cover {s in ACTIVE, p in PROD}:
    Build[p] + Emergency[s,p] >= demand[s,p];

# Eta bounds the recourse cost of every active scenario.
subject to WorstRecourse {s in ACTIVE}:
    Eta >= sum {p in PROD} emergency_cost[p] * Emergency[s,p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ALL"] = ["A", "B", "C", "D"]
ampl.set["PROD"] = ["p1", "p2"]
ampl.param["demand"] = {
    ("A", "p1"): 60, ("A", "p2"): 60, ("B", "p1"): 100, ("B", "p2"): 40,
    ("C", "p1"): 40, ("C", "p2"): 100, ("D", "p1"): 80, ("D", "p2"): 80,
}
ampl.param["capacity_cost"] = {"p1": 2, "p2": 2}
ampl.param["emergency_cost"] = {"p1": 3, "p2": 3}

In [5]:
# run.py
active, upper = ["A"], float("inf")
oracle = ("{s in ALL} sum {p in PROD} emergency_cost[p]"
          " * max(demand[s,p] - Build[p], 0)")
for k in range(1, 6):
    ampl.set["ACTIVE"] = active
    ampl.solve(solver="highs", verbose=False)
    lower = ampl.obj["MasterCost"].value()
    first_stage = ampl.get_value("sum {p in PROD} capacity_cost[p] * Build[p]")
    recourse = ampl.get_data(oracle).to_dict()     # exact over the finite set
    worst = max(recourse, key=recourse.get)
    upper = min(upper, first_stage + recourse[worst])
    print(k, ampl.var["Build"].to_dict(), lower, upper, worst, recourse[worst])
    if upper - lower <= 1e-6:
        break
    active.append(worst)

ampl.set["ACTIVE"] = ampl.set["ALL"].to_list()      # full deterministic equivalent
ampl.solve(solver="highs", verbose=False)
print("all scenarios:", ampl.obj["MasterCost"].value())

1 {'p1': 60, 'p2': 60} 240.0 360 B 120
2 {'p1': 80, 'p2': 40} 300.0 360 C 180
3 {'p1': 40, 'p2': 40} 340.0 360 D 240
4 {'p1': 60, 'p2': 60} 360.0 360 B 120
all scenarios: 360.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; the loop uses `let`, `printf` and `break`:

In [6]:
%%writefile data.dat
set ALL := A B C D;
set PROD := p1 p2;
param demand:  p1   p2 :=
  A            60   60
  B           100   40
  C            40  100
  D            80   80;
param capacity_cost := p1 2  p2 2;
param emergency_cost := p1 3  p2 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
param lower;                   # restricted master bound
param upper default Infinity;  # best worst-case cost found
param recourse {ALL};          # exact recourse cost per scenario
param worst symbolic;          # most costly scenario this round
option solver highs, solver_msg 0;
let ACTIVE := {first(ALL)};
for {k in 1..5} {
    solve;
    let lower := MasterCost;
    let {s in ALL} recourse[s] := sum {p in PROD}
        emergency_cost[p] * max(demand[s,p] - Build[p], 0);
    let worst := first(ALL);
    for {s in ALL} {
        if recourse[s] > recourse[worst] then let worst := s;
    }
    let upper := min(upper,
        sum {p in PROD} capacity_cost[p] * Build[p] + recourse[worst]);
    printf "%d  Build %g %g  lower %g  upper %g  worst %s %g\n",
        k, Build['p1'], Build['p2'], lower, upper, worst, recourse[worst];
    if upper - lower <= 1e-6 then break;
    let ACTIVE := ACTIVE union {worst};
}
let ACTIVE := ALL;
solve;
display MasterCost;

1  Build 60 60  lower 240  upper 360  worst B 120
2  Build 80 40  lower 300  upper 360  worst C 180
3  Build 40 40  lower 340  upper 360  worst D 240
4  Build 60 60  lower 360  upper 360  worst B 120
MasterCost = 360

